# 🖼️ Image Clustering with Hierarchical Clustering
Interactive notebook for exploring the pipeline step-by-step.

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

from cluster import (
    generate_demo_images,
    extract_features,
    reduce_dimensions,
    find_optimal_k,
    cluster_images,
    plot_dendrogram,
    plot_scatter,
    plot_k_sweep
)
print('Imports OK')

## 1. Load / Generate Images

In [ ]:
# Use demo images (or set IMAGE_DIR to your own folder)
IMAGE_DIR = None   # set to 'path/to/your/images' or leave None for demo

if IMAGE_DIR is None:
    image_paths = generate_demo_images('demo_images', n_per_class=25)
else:
    exts = {'.jpg','.jpeg','.png','.bmp','.webp'}
    image_paths = sorted([p for p in Path(IMAGE_DIR).rglob('*') if p.suffix.lower() in exts])

print(f'Loaded {len(image_paths)} images')

# Preview a few
fig, axes = plt.subplots(1, min(6, len(image_paths)), figsize=(14, 3))
for ax, p in zip(axes, image_paths[:6]):
    ax.imshow(Image.open(p))
    ax.set_title(Path(p).stem[:10], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Feature Extraction

In [ ]:
FEATURE_METHOD = 'hog'   # 'hog' | 'pixel' | 'cnn'
features = extract_features(image_paths, method=FEATURE_METHOD)
print(f'Feature matrix: {features.shape}')

## 3. Dimensionality Reduction

In [ ]:
X_pca, X_2d, scaler, pca = reduce_dimensions(features, n_components_pca=30, use_umap=False)
print(f'PCA shape: {X_pca.shape}, 2-D shape: {X_2d.shape}')

## 4. Find Optimal k

In [ ]:
scores = find_optimal_k(X_pca, k_range=(2, 8))
best_k = max(scores, key=lambda k: scores[k]['silhouette'])
print(f'Best k = {best_k}')
plot_k_sweep(scores, best_k, '/tmp/k_sweep.png')
plt.imshow(plt.imread('/tmp/k_sweep.png')); plt.axis('off'); plt.show()

## 5. Cluster & Visualise

In [ ]:
labels, model = cluster_images(X_pca, n_clusters=best_k)
for cl in np.unique(labels):
    print(f'  Cluster {cl}: {np.sum(labels==cl)} images')

plot_scatter(X_2d, labels, '/tmp/scatter.png')
plt.imshow(plt.imread('/tmp/scatter.png')); plt.axis('off'); plt.show()

## 6. Dendrogram

In [ ]:
plot_dendrogram(X_pca, 'ward', '/tmp/dendrogram.png')
plt.imshow(plt.imread('/tmp/dendrogram.png')); plt.axis('off'); plt.show()